# COGS 118D · Fit a model to your own choices

You just played the **Vegas slots** game: a two-armed bandit with two slot machines per round. One machine paid out 70% of the time, the other 30%, and you had to learn which was which. In this notebook you'll fit the models from Lectures 12–15 to **your own choices** and check how well they describe you.

| Step | What you do |
|---|---|
| 1 | Load your data |
| 2 | Look at your behavior: learning curves and win-stay/lose-shift |
| 3 | Fit three models: random responding, win-stay/lose-shift, Rescorla–Wagner |
| 4 | Compare the models (BIC) |
| 5 | Check the winning model: simulate it and compare to your data |
| 6 | *(Instructor)* Fit everyone in the class |

**Before you start:** download your CSV from the end screen of the game (`cogs118d_bandit_XXXXXX.csv`). Then run the cells from top to bottom.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

rng = np.random.default_rng(118)
REWARD_PROBS = [0.3, 0.7]     # option 0 = worse machine, option 1 = better machine

---
## Step 1 · Load your data

Run the cell and pick your CSV file. (Not using Colab? Put the file next to this notebook and set `DATA_PATH` to its name.)

In [ ]:
DATA_PATH = None   # e.g. "cogs118d_bandit_K7Q2XA.csv" if you're not using Colab

if DATA_PATH is None:
    from google.colab import files
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

df = pd.read_csv(DATA_PATH)
df = df[df["chosen_option"].notna()].copy()          # drop trials with no response (rare)
df["chosen_option"] = df["chosen_option"].astype(int)
df["reward"] = df["reward"].astype(int)
print(f"Participant {df['participant_id'].iloc[0]}: {len(df)} trials in {df['block'].nunique()} blocks")
df.head()

Each row is one trial. The columns you'll use:

| Column | What it is |
|---|---|
| `block` | round of the game; each round used a new pair of slot machines |
| `trial` | trial number within the round |
| `choice_type` | `free` (you chose between two slot machines) or `forced` (only one slot machine was shown) |
| `chosen_option` | **1** if you chose the better machine (70%), **0** if the worse one (30%) |
| `reward` | 1 if you got a reward, 0 if not |
| `rt_ms` | response time in milliseconds |

The model doesn't need to know which slot machine was on the left: it learns a value for each **slot machine**, so we code choices by option (better or worse).

---
## Step 2 · Look at your behavior

Always look before you model (Lecture 13). Two model-independent summaries:

1. **Learning curve:** how often you chose the better slot machine as each round went on.
2. **Win-stay/lose-shift:** how often you repeated your previous choice after a reward vs. after no reward.

In [ ]:
def stay_rates(d):
    """p(stay | previous loss) and p(stay | previous win), using consecutive free-choice trials within each block."""
    stays = {0: [], 1: []}
    for _, b in d.groupby("block"):
        b = b.sort_values("trial")
        for prev, cur in zip(b.iloc[:-1].itertuples(), b.iloc[1:].itertuples()):
            if prev.choice_type == "free" and cur.choice_type == "free":
                stays[prev.reward].append(cur.chosen_option == prev.chosen_option)
    return np.mean(stays[0]) if stays[0] else np.nan, np.mean(stays[1]) if stays[1] else np.nan

free = df[df["choice_type"] == "free"]
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [2, 1]})
for blk, b in free.groupby("block"):
    axes[0].plot(b["trial"], b["chosen_option"].rolling(10, min_periods=1).mean(), lw=2.5, label=f"round {blk}")
axes[0].axhline(0.5, color="gray", ls="--", lw=1)
axes[0].set_xlabel("trial"); axes[0].set_ylabel("p(chose the better slot machine)\n(10-trial running average)")
axes[0].set_ylim(0, 1.05); axes[0].legend(); axes[0].set_title("Your learning curves")
lose_stay, win_stay = stay_rates(df)
axes[1].plot([0, 1], [lose_stay, win_stay], "o-", ms=12, lw=3, color="black")
axes[1].set_xticks([0, 1], ["after no reward", "after reward"]); axes[1].set_xlim(-0.3, 1.3); axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("p(repeat your last choice)"); axes[1].set_title("Your stay behavior")
plt.tight_layout(); plt.show()
print(f"You chose the better slot machine on {free['chosen_option'].mean():.0%} of free choices.")
print(f"p(stay | no reward) = {lose_stay:.2f},   p(stay | reward) = {win_stay:.2f}")

✍️ **Think about it.** Compare your stay plot with the five model fingerprints from Lecture 13 (Part B of Homework 4). Which model does your pattern look most like?

---
## Step 3 · Fit three models to your choices

These are three of the five models from Wilson & Collins (2019), written for this task. Two details are specific to the game:

- **Each round starts fresh.** The slot machines change, so the values reset to 0.5 at the start of every block.
- **Forced-choice trials** (if your game had any) aren't your choice, so they don't count toward the likelihood, but you still **learn** from their outcomes.

The likelihood functions are the same idea as Homework 4: run the model through your actual choices and rewards, and add up −log of the probability it gave to each choice you made.

In [ ]:
def lik_M1random(d, b):
    """Random responding: choose the better slot machine with probability b, ignoring rewards."""
    f = d[d["choice_type"] == "free"]
    p = np.where(f["chosen_option"] == 1, b, 1 - b)
    return -np.sum(np.log(np.clip(p, 1e-10, 1)))


def lik_M2WSLS(d, epsilon):
    """Noisy win-stay/lose-shift: repeat after a reward, switch after no reward; random with probability epsilon."""
    nll = 0.0
    for _, blk in d.groupby("block"):
        last_a, last_r = None, None
        for row in blk.sort_values("trial").itertuples():
            if row.choice_type == "free":
                if last_a is None:
                    p_choice = 0.5
                else:
                    stay = (row.chosen_option == last_a)
                    p_stay = 1 - epsilon / 2 if last_r == 1 else epsilon / 2
                    p_choice = p_stay if stay else 1 - p_stay
                nll -= np.log(max(p_choice, 1e-10))
            last_a, last_r = row.chosen_option, row.reward
    return nll


def lik_M3RescorlaWagner(d, alpha, beta):
    """Rescorla-Wagner with softmax. Values reset at the start of each block; forced trials update but don't count."""
    nll = 0.0
    for _, blk in d.groupby("block"):
        Q = np.array([0.5, 0.5])
        for row in blk.sort_values("trial").itertuples():
            a, r = row.chosen_option, row.reward
            if row.choice_type == "free":
                p = np.exp(beta * Q) / np.sum(np.exp(beta * Q))
                nll -= np.log(max(p[a], 1e-10))
            Q[a] = Q[a] + alpha * (r - Q[a])
    return nll


def fit(lik, d, bounds, n_starts=10, rng=rng):
    """Minimize the negative log-likelihood from several random starting points; keep the best."""
    best = None
    for s in range(n_starts):
        x0 = [rng.uniform(lo, min(hi, 10)) for lo, hi in bounds]
        res = minimize(lambda x: lik(d, *x), x0, method="L-BFGS-B", bounds=bounds)
        if best is None or res.fun < best.fun:
            best = res
    return best.x, best.fun


n_free = int((df["choice_type"] == "free").sum())
fits = {}
fits["M1: random"] = fit(lik_M1random, df, [(0.001, 0.999)])
fits["M2: WSLS"] = fit(lik_M2WSLS, df, [(0.001, 0.999)])
fits["M3: Rescorla-Wagner"] = fit(lik_M3RescorlaWagner, df, [(0, 1), (0, 30)])

table = pd.DataFrame([{"model": m, "parameters": np.round(x, 3), "k": len(x), "NLL": round(nll, 2),
                       "BIC": round(2 * nll + len(x) * np.log(n_free), 2)} for m, (x, nll) in fits.items()])
print(f"Free choices: {n_free}.  Coin flip on every trial: NLL = {n_free * np.log(2):.2f}")
table

In [ ]:
alpha_hat, beta_hat = fits["M3: Rescorla-Wagner"][0]
print(f"Your learning rate:          alpha = {alpha_hat:.2f}")
print(f"Your inverse temperature:    beta  = {beta_hat:.2f}")
if beta_hat > 29.5:
    print("Note: beta hit the upper bound (30). You chose very consistently, so the data only say 'beta is large' (Lecture 14).")

**Reading the fits.**

- **NLL** (negative log-likelihood): lower = the model makes your choices more probable. Compare it with the coin-flip value printed above.
- **BIC** = 2 × NLL + k × ln(number of choices). It adds a penalty for each parameter, so a model has to earn its extra parameters (Lecture 15). **Lower BIC = better.**
- **α**: how much one outcome moves your value for a slot machine. **β**: how consistently you chose the slot machine you valued more.

✍️ **Think about it.** Which model has the lowest BIC for you? Does that match your guess from the stay plot? With only ~60 choices, how much would you trust your α (Lecture 14, parameter recovery)?

---
## Step 4 · Does the fitted model behave like you?

A model can win the BIC comparison and still miss important features of your behavior. The check (Rule 8, Lecture 15, a **posterior predictive check**): simulate the fitted Rescorla–Wagner model **on the same game you played** many times, and compare its learning curve and stay behavior with yours.

In [ ]:
def simulate_RW_on_my_game(d, alpha, beta, rng):
    """Play your exact sequence of trials (same blocks, same forced trials) with the fitted RW model."""
    rows = []
    for blk_id, blk in d.groupby("block"):
        Q = np.array([0.5, 0.5])
        for row in blk.sort_values("trial").itertuples():
            if row.choice_type == "free":
                p = np.exp(beta * Q) / np.sum(np.exp(beta * Q))
                a = int(rng.random() < p[1])
            else:
                a = row.chosen_option                       # forced trials: same slot machine you were shown
            r = int(rng.random() < REWARD_PROBS[a])
            Q[a] = Q[a] + alpha * (r - Q[a])
            rows.append({"block": blk_id, "trial": row.trial, "choice_type": row.choice_type, "chosen_option": a, "reward": r})
    return pd.DataFrame(rows)

n_sims = 300
sims = [simulate_RW_on_my_game(df, alpha_hat, beta_hat, rng) for _ in range(n_sims)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), gridspec_kw={"width_ratios": [2, 1]})
sim_curve = pd.concat(sims).query("choice_type == 'free'").groupby("trial")["chosen_option"].mean()
my_curve = free.groupby("trial")["chosen_option"].mean().rolling(5, min_periods=1, center=True).mean()
axes[0].plot(sim_curve.index, sim_curve.values, color="purple", lw=3, label=f"fitted RW model ({n_sims} simulations)")
axes[0].plot(my_curve.index, my_curve.values, color="black", lw=2, label="you (averaged over rounds, smoothed)")
axes[0].axhline(0.5, color="gray", ls="--", lw=1); axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("trial"); axes[0].set_ylabel("p(chose the better slot machine)"); axes[0].legend(); axes[0].set_title("Learning curve")
sim_stay = np.array([stay_rates(s) for s in sims])
axes[1].plot([0, 1], np.nanmean(sim_stay, 0), "o-", ms=12, lw=3, color="purple", label="fitted RW model")
lo, hi = np.nanpercentile(sim_stay, [2.5, 97.5], axis=0)
axes[1].vlines([0, 1], lo, hi, color="purple", lw=6, alpha=0.3)
axes[1].plot([0, 1], [lose_stay, win_stay], "o-", ms=12, lw=3, color="black", label="you")
axes[1].set_xticks([0, 1], ["after no reward", "after reward"]); axes[1].set_xlim(-0.3, 1.3); axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("p(repeat your last choice)"); axes[1].legend(loc="lower right"); axes[1].set_title("Stay behavior (bars: 95% of simulations)")
plt.tight_layout(); plt.show()

✍️ **Think about it.** Does the fitted model reproduce your learning curve? Do your stay rates fall inside the model's 95% range? If not, what does the model get wrong about you, and which other model (for example, adding a choice kernel, M5) might fix it?

---
## Step 5 *(instructor)* · Fit everyone in the class

The game sends every student's anonymous data to the class Google Sheet. To analyze it: in the sheet, **File → Download → Comma-separated values (.csv)** on the `responses` tab, then upload that file here. Each student is identified only by their random ID.

In [ ]:
CLASS_PATH = None   # e.g. "bandit_class_data.csv"; leave None to upload in Colab

if CLASS_PATH is None:
    from google.colab import files
    CLASS_PATH = list(files.upload().keys())[0]

everyone = pd.read_csv(CLASS_PATH)
everyone = everyone[everyone["chosen_option"].notna()].copy()
everyone["chosen_option"] = everyone["chosen_option"].astype(int)
everyone["reward"] = everyone["reward"].astype(int)
# if a student's data arrived twice (e.g. they pressed "Try again"), keep the first copy
everyone = everyone.drop_duplicates(subset=["participant_id", "block", "trial"])

rows = []
for pid, d in everyone.groupby("participant_id"):
    (a_hat, b_hat), nll_rw = fit(lik_M3RescorlaWagner, d, [(0, 1), (0, 30)], n_starts=5)
    (eps,), nll_wsls = fit(lik_M2WSLS, d, [(0.001, 0.999)], n_starts=3)
    n = int((d["choice_type"] == "free").sum())
    rows.append({"participant_id": pid, "n_free": n, "p_better": d.loc[d.choice_type == "free", "chosen_option"].mean(),
                 "alpha": a_hat, "beta": b_hat,
                 "BIC_RW": 2 * nll_rw + 2 * np.log(n), "BIC_WSLS": 2 * nll_wsls + np.log(n), "BIC_random": 2 * fit(lik_M1random, d, [(0.001, 0.999)], n_starts=2)[1] + np.log(n)})
class_fits = pd.DataFrame(rows)
class_fits["best_model"] = class_fits[["BIC_RW", "BIC_WSLS", "BIC_random"]].idxmin(axis=1).str.replace("BIC_", "")
print(f"{len(class_fits)} participants")
print(class_fits["best_model"].value_counts().to_string())
class_fits.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(class_fits["alpha"], bins=np.linspace(0, 1, 11), color="#D62728", edgecolor="white"); axes[0].set_xlabel("learning rate α")
axes[1].hist(class_fits["beta"], bins=np.linspace(0, 30, 16), color="#D62728", edgecolor="white"); axes[1].set_xlabel("inverse temperature β")
axes[2].scatter(class_fits["beta"], class_fits["alpha"], color="#9467BD", s=60)
axes[2].set_xlabel("β"); axes[2].set_ylabel("α"); axes[2].set_title("each dot is one student")
axes[0].set_ylabel("number of students")
plt.tight_layout(); plt.show()